# Fase 6 — Proceso Gaussiano: mapas de riesgo con incertidumbre

Quinta fase del pipeline. Toma las probabilidades por celda del Transformer (notebook 05) y ajusta un Proceso Gaussiano espacial (kernel Matérn-3/2) para producir:

1. Un campo continuo de riesgo (μ): suaviza las probabilidades discretas por celda en una superficie continua.
2. Un campo de incertidumbre (σ): bandas de confianza bayesianas.

Flujo: reconstrucción de la rejilla (catálogo cacheado) → carga de las predicciones del notebook 05 → calibración isotónica → GP por región → mapas a 300 DPI.

Como el modelo espacial iguala a la climatología de Poisson y no hay señal temporal a 30 días, el resultado es un mapa de peligro estático (qué zonas son crónicamente más peligrosas), no un pronóstico temporal. La aportación del GP es cuantificar la incertidumbre de esa cartografía.

In [ ]:
# Entorno + NB_TAG (derivado automaticamente del NOMBRE de este notebook)
import os

try:
    from google.colab import drive

    # Si el import funciona, el entorno es Colab
    print("Entorno Colab detectado: Montando Google Drive...")
    estoy_drive = True
    drive.mount('/content/drive')

    import sys
    sys.path.append("/content/drive/MyDrive/VIU/TFM/code")
    sys.path.append("/content/drive/MyDrive/VIU/TFM/results")
    sys.path.append("/content/drive/MyDrive/VIU/TFM/USGS")

    # Nombre del notebook en Colab -> NB_TAG = primera parte antes del '_'
    from google.colab import _message
    _meta = _message.blocking_request("get_ipynb")["ipynb"]["metadata"]
    _nb_name = _meta.get("colab", {}).get("name", "05a.ipynb")
    NB_TAG = os.path.splitext(_nb_name)[0].split("_")[0]

except ImportError:
    estoy_drive = False
    # Entorno local (VS Code / Jupyter): nombre del fichero -> NB_TAG
    try:
        NB_TAG = os.path.splitext(os.path.basename(__vsc_ipynb_file__))[0].split("_")[0]
    except NameError:
        NB_TAG = "05a"

print(estoy_drive)
print(f"NB_TAG = {NB_TAG}")

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from tqdm import tqdm

!pip install -q seisbench

sys.path.append(os.path.abspath(""))

# Reproducibilidad (semillas fijas)
SEED = 42
np.random.seed(SEED)

## 1. Datos: catálogo y declustering

In [ ]:
# --- Configuración: lista de regiones (descarga concatenada desde USGS FDSN) ---
#
# Se admite una lista de regiones (California y Alaska): cada bbox se descarga por
# separado y se concatena. El grafo respeta los grupos geográficos (no se conectan
# celdas entre regiones distantes).

REGIONS = [
    {"name": "california", "minlatitude":  32.0, "maxlatitude":  42.0,
                           "minlongitude": -124.0, "maxlongitude": -114.0},
    {"name": "alaska",     "minlatitude":  51.0, "maxlatitude":  72.0,
                           "minlongitude": -180.0, "maxlongitude": -130.0},
]

# Otras posibles regiones (descomenta para añadir):
#  {"name": "japan",  "minlatitude":  24.0, "maxlatitude":  46.0, "minlongitude":  122.0, "maxlongitude":  146.0},
#  {"name": "italy",  "minlatitude":  35.0, "maxlatitude":  48.0, "minlongitude":    6.0, "maxlongitude":   19.0},
#  {"name": "chile",  "minlatitude": -45.0, "maxlatitude": -17.0, "minlongitude":  -76.0, "maxlongitude":  -66.0},

REGIONS_TAG       = "+".join(r["name"] for r in REGIONS)
STARTTIME         = "2000-01-01"
ENDTIME           = "2024-12-31"
MIN_MAG_CATALOGO  = 2.5
MIN_MAG_TARGET    = 4.5
GRID_DEG          = 0.5
PREDICTION_WINDOW = 30
LOOKBACK          = 60

# Bbox global para el grid (minimum enclosing box de las regiones)
REGION = {
    "minlatitude":  min(r["minlatitude"]  for r in REGIONS),
    "maxlatitude":  max(r["maxlatitude"]  for r in REGIONS),
    "minlongitude": min(r["minlongitude"] for r in REGIONS),
    "maxlongitude": max(r["maxlongitude"] for r in REGIONS),
}

# --- Rutas ---
if estoy_drive:
  # Rutas absolutas a Google Drive
  BASE_DIR = "/content/drive/MyDrive/VIU/TFM"

  RESULTS_DIR = os.path.join(BASE_DIR, "results")
  FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
  USGS_DIR    = os.path.join(BASE_DIR, "USGS")
else:
  RESULTS_DIR = os.path.join("..", "results")
  FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
  USGS_DIR    = os.path.join("..", "USGS")

for d in (RESULTS_DIR, FIGURES_DIR, USGS_DIR):
    os.makedirs(d, exist_ok = True)

# --- Cargar/Descargar catálogos por región y concatenar ---
import urllib.parse
import urllib.error

def fetch_region_catalog(region):
    """Carga del cache o descarga el catálogo USGS para una región."""
    path = os.path.join(USGS_DIR,
        f"{region['name']}_{STARTTIME[:4]}_{ENDTIME[:4]}_M{MIN_MAG_CATALOGO}.csv")
    if os.path.exists(path):
        print(f"  [{region['name']}] caché: {path}")
        df = pd.read_csv(path)
        df["time"] = pd.to_datetime(df["time"], format = "ISO8601", utc = True)
        return df

    print(f"  [{region['name']}] descargando desde USGS FDSN...")
    BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    fechas_start = pd.date_range(STARTTIME, ENDTIME, freq = "MS")
    fechas_end   = list(fechas_start[1:]) + [pd.to_datetime(ENDTIME) + pd.Timedelta(days = 1)]

    dfs = []
    for inicio, fin in tqdm(list(zip(fechas_start, fechas_end)),
                            desc = f"  USGS {region['name']}"):
        s, e = inicio.strftime("%Y-%m-%d"), fin.strftime("%Y-%m-%d")
        params = {
            "format": "csv", "starttime": s, "endtime": e,
            "minmagnitude": MIN_MAG_CATALOGO,
            "minlatitude":  region["minlatitude"],  "maxlatitude":  region["maxlatitude"],
            "minlongitude": region["minlongitude"], "maxlongitude": region["maxlongitude"],
            "orderby": "time-asc",
        }
        url = BASE_URL + "?" + urllib.parse.urlencode(params)
        try:
            df_chunk = pd.read_csv(url)
        except (pd.errors.EmptyDataError, urllib.error.HTTPError):
            df_chunk = pd.DataFrame()
        dfs.append(df_chunk)

    df = pd.concat(dfs, ignore_index = True)
    df["time"] = pd.to_datetime(df["time"], format = "ISO8601", utc = True)
    df = (df.drop_duplicates(subset = ["id"])
            .sort_values("time")
            .reset_index(drop = True))
    df.to_csv(path, index = False)
    print(f"  [{region['name']}] guardado: {path}")
    return df


print(f"Cargando {len(REGIONS)} región(es)...")
dfs_per_region = []
for region in REGIONS:
    df_r = fetch_region_catalog(region)
    df_r["region"] = region["name"]
    dfs_per_region.append(df_r)
    print(f"    └─ {region['name']}: {len(df_r):,} eventos M≥{MIN_MAG_CATALOGO}, "
          f"{(df_r['mag'] >= MIN_MAG_TARGET).sum()} M≥{MIN_MAG_TARGET}")

df_cat = pd.concat(dfs_per_region, ignore_index = True).sort_values("time").reset_index(drop = True)
df_cat["date"] = df_cat["time"].dt.floor("D")

print(f"\n=== Configuración ===")
print(f"Regiones:          {REGIONS_TAG}  ({len(REGIONS)} bbox)")
print(f"Bbox global:       lat [{REGION['minlatitude']}, {REGION['maxlatitude']}], "
      f"lon [{REGION['minlongitude']}, {REGION['maxlongitude']}]")
print(f"Ventana temporal:  {STARTTIME} → {ENDTIME}")
print(f"M catálogo ≥ {MIN_MAG_CATALOGO}  |  M objetivo ≥ {MIN_MAG_TARGET}")
print(f"Grid espacial:     {GRID_DEG}° × {GRID_DEG}°  ({int((REGION['maxlatitude']-REGION['minlatitude'])/GRID_DEG)} × "
      f"{int((REGION['maxlongitude']-REGION['minlongitude'])/GRID_DEG)} celdas máximas)")
print(f"PREDICTION_WINDOW: {PREDICTION_WINDOW} días")
print(f"LOOKBACK:          {LOOKBACK} días")
print()
print(f"=== Catálogo combinado ===")
print(f"Total eventos:     {len(df_cat):,}")
print(f"Rango temporal:    {df_cat['time'].min()} → {df_cat['time'].max()}")
print(f"Rango magnitud:    {df_cat['mag'].min():.1f} → {df_cat['mag'].max():.1f}")
print(f"Eventos M ≥ {MIN_MAG_TARGET}:  {(df_cat['mag'] >= MIN_MAG_TARGET).sum():,}  "
      f"(~{(df_cat['mag'] >= MIN_MAG_TARGET).sum() / 15.0:.1f}/año)")
print()
print(f"Por región:")
print(df_cat.groupby("region").agg(
    n_total      = ("mag", "size"),
    n_M_target   = ("mag", lambda m: (m >= MIN_MAG_TARGET).sum()),
).to_string())

In [ ]:
# --- Declustering de Gardner-Knopoff (1974) ---
# Identifica mainshocks INDEPENDIENTES (filtra aftershocks/foreshocks dependientes).
# Tras el filtrado quedan los mainshocks independientes, no las secuencias de réplicas.
#
# Ventanas (van Stiphout et al. 2012):
#   L(km)   = 10^(0.1238·M + 0.983)
#   T(días) = 10^(0.5409·M - 0.547)        si M ≤ 6.5
#   T(días) = 10^(0.032·M + 2.7389)        si M >  6.5

def gardner_knopoff_window(M):
    R_km = 10 ** (0.1238 * M + 0.983)
    if M <= 6.5:
        T_days = 10 ** (0.5409 * M - 0.547)
    else:
        T_days = 10 ** (0.032 * M + 2.7389)
    return T_days, R_km

def haversine_km(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371.0
    lat1r = np.radians(lat1)
    lat2r = np.radians(lat2_arr)
    dlat  = lat2r - lat1r
    dlon  = np.radians(lon2_arr - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df_target_pool = (df_cat[df_cat["mag"] >= MIN_MAG_TARGET]
                    .sort_values("time")
                    .reset_index(drop = True))
times_sec = (df_target_pool["time"] - df_target_pool["time"].iloc[0]).dt.total_seconds().values
lats      = df_target_pool["latitude"].values
lons      = df_target_pool["longitude"].values
mags      = df_target_pool["mag"].values
n         = len(df_target_pool)

is_cluster  = np.zeros(n, dtype = bool)
sort_by_mag = np.argsort(-mags)

for i in sort_by_mag:
    if is_cluster[i]:
        continue
    M_main       = mags[i]
    T_days, R_km = gardner_knopoff_window(M_main)
    T_sec        = T_days * 86400.0
    dt           = np.abs(times_sec - times_sec[i])
    candidates   = np.where((dt <= T_sec) & (dt > 0) & (mags < M_main) & (~is_cluster))[0]
    if len(candidates) == 0:
        continue
    dist        = haversine_km(lats[i], lons[i], lats[candidates], lons[candidates])
    cluster_idx = candidates[dist <= R_km]
    is_cluster[cluster_idx] = True

df_targets = df_target_pool[~is_cluster].copy()
df_targets["date"] = df_targets["time"].dt.floor("D")

print(f"=== Declustering Gardner-Knopoff (1974) ===")
print(f"Eventos M ≥ {MIN_MAG_TARGET} originales:        {n}")
print(f"Mainshocks independientes:        {len(df_targets)}  ({100 * len(df_targets) / n:.1f}%)")
print(f"Aftershocks/foreshocks eliminados: {n - len(df_targets)}")
print(f"Tasa anual de mainshocks:         {len(df_targets) / 15.0:.2f}/año")
print()
print(f"(Debe coincidir con los 71 mainshocks reportados por el nb 03)")

## 2. Discretización y centroides

In [ ]:
# --- Discretización espacial: grid 0.5° × 0.5° y filtrado de celdas activas ---
# Cada evento se asigna a una celda. Las celdas con muy pocos eventos se descartan
# (más ruido que señal).
# También se asigna a cada celda su REGIÓN (california/alaska) para mantener la
# separación lógica en el grafo y el cálculo de b-value.

# Crear ejes del grid sobre el bbox GLOBAL
lat_edges = np.arange(REGION["minlatitude"],  REGION["maxlatitude"]  + 1e-9, GRID_DEG)
lon_edges = np.arange(REGION["minlongitude"], REGION["maxlongitude"] + 1e-9, GRID_DEG)
n_lat = len(lat_edges) - 1
n_lon = len(lon_edges) - 1

# Asignar cada evento del catálogo a su celda
df_cat["lat_idx"] = np.clip(np.digitize(df_cat["latitude"],  lat_edges) - 1, 0, n_lat - 1)
df_cat["lon_idx"] = np.clip(np.digitize(df_cat["longitude"], lon_edges) - 1, 0, n_lon - 1)
df_cat["cell_id"] = df_cat["lat_idx"] * n_lon + df_cat["lon_idx"]

# El declustering ya generó df_targets; se replica la asignación a celda
df_targets["lat_idx"] = np.clip(np.digitize(df_targets["latitude"],  lat_edges) - 1, 0, n_lat - 1)
df_targets["lon_idx"] = np.clip(np.digitize(df_targets["longitude"], lon_edges) - 1, 0, n_lon - 1)
df_targets["cell_id"] = df_targets["lat_idx"] * n_lon + df_targets["lon_idx"]

# --- Filtrado de celdas activas ---
MIN_EVENTOS_CELDA = 30

events_per_cell = df_cat.groupby("cell_id").size()
active_cells    = events_per_cell[events_per_cell >= MIN_EVENTOS_CELDA].index.values
active_cells    = np.sort(active_cells)
n_active        = len(active_cells)

cell_id_to_idx = {cid: i for i, cid in enumerate(active_cells)}

# --- Asignar cada celda activa a su región (la que mayoritariamente contiene sus eventos) ---
def cell_region(cell_id):
    """Devuelve el nombre de la región que contiene los eventos de esta celda."""
    region_counts = df_cat[df_cat["cell_id"] == cell_id]["region"].value_counts()
    return region_counts.index[0] if len(region_counts) else None

cell_region_arr = np.array([cell_region(cid) for cid in active_cells])

# Sanity check: cada celda activa debe tener al menos una región
assert all(r is not None for r in cell_region_arr), "Hay celdas activas sin región asignada"

# Mainshocks dentro vs fuera de celdas activas
mainshocks_active  = df_targets[df_targets["cell_id"].isin(active_cells)].copy()
mainshocks_outside = df_targets[~df_targets["cell_id"].isin(active_cells)].copy()

print(f"=== Grid espacial ===")
print(f"Bbox global:     lat [{REGION['minlatitude']}, {REGION['maxlatitude']}], "
      f"lon [{REGION['minlongitude']}, {REGION['maxlongitude']}]")
print(f"Tamaño del grid: {n_lat} × {n_lon} = {n_lat * n_lon} celdas máximas")
print(f"Celdas con ≥ 1 evento:                {(events_per_cell > 0).sum()}")
print(f"Celdas activas (≥ {MIN_EVENTOS_CELDA} eventos M≥{MIN_MAG_CATALOGO}):  {n_active}")
print()
print(f"Distribución de celdas activas por región:")
unique, counts = np.unique(cell_region_arr, return_counts = True)
for r, c in zip(unique, counts):
    print(f"  {r:12s}  {c} celdas")
print()
print(f"=== Cobertura de mainshocks ===")
print(f"Mainshocks totales (post G-K):        {len(df_targets)}")
print(f"  ├─ dentro de celdas activas:        {len(mainshocks_active)}  "
      f"({100*len(mainshocks_active)/len(df_targets):.1f}%)")
print(f"  └─ fuera (en celdas con baja sismicidad): {len(mainshocks_outside)}")
print()
print(f"Distribución de eventos en celdas activas:")
print(f"  median: {int(np.median(events_per_cell[active_cells]))} eventos/celda")
print(f"  min:    {int(np.min(events_per_cell[active_cells]))} eventos/celda")
print(f"  max:    {int(np.max(events_per_cell[active_cells]))} eventos/celda")
print()
if len(mainshocks_outside) > 0:
    print(f"⚠️  Mainshocks fuera de celdas activas (primeros 10):")
    print(mainshocks_outside[["time", "latitude", "longitude", "mag", "place"]].head(10).to_string(index = False))

In [ ]:
# --- Centroides (lat, lon) de las celdas activas ---
cell_centroids = np.zeros((n_active, 2), dtype = np.float32)
for ci, cid in enumerate(active_cells):
    li, lo = cid // n_lon, cid % n_lon
    cell_centroids[ci, 0] = (lat_edges[li] + lat_edges[li + 1]) / 2   # lat centro
    cell_centroids[ci, 1] = (lon_edges[lo] + lon_edges[lo + 1]) / 2   # lon centro
print(f"Centroides calculados: {cell_centroids.shape}  (n_celdas, [lat, lon])")

## 3. Predicciones y calibración

In [ ]:
# --- Cargar predicciones del Transformer (nb05) desde el pickle ---
import pickle, glob

MODELO_FUENTE = "05"   # las predicciones vienen del nb05 (Transformer)
cands = sorted(glob.glob(os.path.join(RESULTS_DIR, "*multiseed_results.pkl")))
print("Pickles multi-seed disponibles:", [os.path.basename(c) for c in cands])
PKL = next((c for c in cands if os.path.basename(c).startswith(MODELO_FUENTE)),
           cands[0] if cands else None)
assert PKL is not None, "No se encontro ningun pickle *multiseed_results.pkl en results/"
print(f"Usando: {os.path.basename(PKL)}")

d = pickle.load(open(PKL, "rb"))
ensemble_logits = np.asarray(d["ensemble_logits"], dtype=float)
test_y          = np.asarray(d["test_y"], dtype=int)

# Reshape de (dias*celdas,) a (dias, celdas) usando n_active del grid reconstruido
n_test_days = ensemble_logits.size // n_active
assert ensemble_logits.size == n_test_days * n_active, \
    f"n_active ({n_active}) no cuadra con el pickle ({ensemble_logits.size}). " \
    f"Revisa que el grid/MIN_EVENTOS_CELDA coincide con el del nb05."
scores_2d = ensemble_logits.reshape(n_test_days, n_active)
y_2d      = test_y.reshape(n_test_days, n_active)
print(f"Predicciones: {n_test_days} dias de test x {n_active} celdas")

In [ ]:
# --- Calibracion isotonica: logits -> probabilidades reales ---
# El GP necesita probabilidades, no logits crudos. La regresion isotonica mapea
# los scores del modelo a frecuencias empíricas reales (regresión isotónica).
# La isotónica se ajusta sobre todo el test (uso descriptivo, para construir el mapa
# del periodo de test). La validación de la calibración sin circularidad (split-half)
# está en la celda siguiente.
from sklearn.isotonic import IsotonicRegression

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(ensemble_logits, test_y)
prob_cal = iso.predict(ensemble_logits).reshape(n_test_days, n_active)

# Riesgo por celda = probabilidad calibrada media sobre el periodo de test
risk_per_cell = prob_cal.mean(axis=0)   # (n_active,)
print(f"Riesgo por celda (calibrado):  min={risk_per_cell.min():.4f}  "
      f"media={risk_per_cell.mean():.4f}  max={risk_per_cell.max():.4f}")
print(f"(comparar con base_rate del problema ~{y_2d.mean():.4f})")

In [ ]:
# --- Validacion de la calibracion: curva de fiabilidad + Brier (split-half honesto) ---
# Para evitar circularidad, la isotónica se ajusta en una mitad del test y la curva
# y el Brier se evalúan en la otra mitad.
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

rng  = np.random.default_rng(42)
perm = rng.permutation(ensemble_logits.size)
half = perm.size // 2
iA, iB = perm[:half], perm[half:]

prob_raw   = 1.0 / (1.0 + np.exp(-ensemble_logits))                       # sin calibrar
iso_half   = IsotonicRegression(out_of_bounds="clip").fit(ensemble_logits[iA], test_y[iA])
prob_cal_B = iso_half.predict(ensemble_logits[iB])

brier_raw = brier_score_loss(test_y[iB], prob_raw[iB])
brier_cal = brier_score_loss(test_y[iB], prob_cal_B)
print(f"Brier score (mitad B):  sin calibrar = {brier_raw:.4f}   isotonica = {brier_cal:.4f}")
print(f"  -> menor Brier = mejor calibracion  ({'MEJORA' if brier_cal < brier_raw else 'no mejora'})")

fr_raw, mp_raw = calibration_curve(test_y[iB], prob_raw[iB],   n_bins=10, strategy="quantile")
fr_cal, mp_cal = calibration_curve(test_y[iB], prob_cal_B,     n_bins=10, strategy="quantile")

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="calibracion perfecta")
ax.plot(mp_raw, fr_raw, "o-", color="gray",    label=f"sin calibrar (Brier={brier_raw:.4f})")
ax.plot(mp_cal, fr_cal, "s-", color="darkred", label=f"isotonica (Brier={brier_cal:.4f})")
ax.set_xlabel("Probabilidad predicha"); ax.set_ylabel("Frecuencia observada")
ax.set_title("Diagrama de calibracion (split-half del test)", fontweight="bold")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout()
cpath = os.path.join(FIGURES_DIR, f"{NB_TAG}_calibracion.png")
plt.savefig(cpath, dpi=300, bbox_inches="tight"); plt.show()
print(f"Figura: {cpath}")

## 4. Proceso Gaussiano y mapas

In [ ]:
# --- Gaussian Process espacial POR REGION (Matern-3/2) ---
# California y Alaska estan a >3000 km: un GP por region (no interpolar el hueco).
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel

GRID_FINO = 90   # resolucion de la rejilla de prediccion por eje

gp_results = {}
for region in np.unique(cell_region_arr):
    mask = cell_region_arr == region
    X = cell_centroids[mask]          # (n_cells_r, 2) [lat, lon]
    y = risk_per_cell[mask]

    kernel = (C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(0.3, 10.0), nu=1.5)
              + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e-1)))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                  n_restarts_optimizer=4, random_state=42)
    gp.fit(X, y)

    # Rejilla fina sobre el bbox de la region (con un pequeno margen)
    la = np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, GRID_FINO)
    lo = np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, GRID_FINO)
    LA, LO = np.meshgrid(la, lo)
    Xg = np.column_stack([LA.ravel(), LO.ravel()])
    mu, sd = gp.predict(Xg, return_std=True)
    mu = np.clip(mu, 0.0, None)        # el riesgo no puede ser negativo

    gp_results[region] = dict(LA=LA, LO=LO, mu=mu.reshape(LA.shape),
                              sd=sd.reshape(LA.shape), X=X, y=y)
    print(f"{region:12s}  {mask.sum():3d} celdas  |  kernel ajustado: {gp.kernel_}")

In [ ]:
# --- Mapas de riesgo (mu) e incertidumbre (sigma) por region, 300 DPI ---
import pandas as pd

# Mainshocks reales del periodo de TEST (para overlay de validacion visual)
VAL_END = "2022-12-31"
df_test_ms = df_targets[df_targets["time"] > pd.Timestamp(VAL_END, tz="UTC")]

for region, R in gp_results.items():
    fig, axs = plt.subplots(1, 2, figsize=(15, 6))

    # (a) Riesgo (GP mu)
    pc = axs[0].pcolormesh(R["LO"], R["LA"], R["mu"], shading="auto", cmap="YlOrRd")
    axs[0].scatter(R["X"][:,1], R["X"][:,0], c="k", s=6, alpha=0.25, label="celdas activas")
    ms = df_test_ms[(df_test_ms["latitude"].between(R["LA"].min(), R["LA"].max())) &
                    (df_test_ms["longitude"].between(R["LO"].min(), R["LO"].max()))]
    axs[0].scatter(ms["longitude"], ms["latitude"], marker="*", c="blue", s=120,
                   edgecolor="white", linewidth=0.6, label="mainshocks reales (test)", zorder=5)
    fig.colorbar(pc, ax=axs[0], label="riesgo calibrado (prob. 30d)")
    axs[0].set_xlabel("Longitud"); axs[0].set_ylabel("Latitud")
    axs[0].set_title(f"Mapa de riesgo (GP mu) - {region}", fontweight="bold")
    axs[0].legend(loc="upper right", fontsize=8)

    # (b) Incertidumbre (GP sigma)
    pc2 = axs[1].pcolormesh(R["LO"], R["LA"], R["sd"], shading="auto", cmap="Blues")
    axs[1].scatter(R["X"][:,1], R["X"][:,0], c="k", s=6, alpha=0.25)
    fig.colorbar(pc2, ax=axs[1], label="incertidumbre (GP sigma)")
    axs[1].set_xlabel("Longitud"); axs[1].set_ylabel("Latitud")
    axs[1].set_title(f"Incertidumbre (GP sigma) - {region}", fontweight="bold")

    plt.suptitle(f"GP riesgo + incertidumbre - {region} (test 2023-2024)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    fpath = os.path.join(FIGURES_DIR, f"{NB_TAG}_mapa_riesgo_gp_{region}.png")
    plt.savefig(fpath, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Figura: {fpath}")

## 5. Validación del mapa

In [ ]:
# --- Validacion cuantitativa del mapa de riesgo (coherencia espacial) ---
from sklearn.metrics import roc_auc_score

# Target espacial: ¿la celda tuvo >= 1 mainshock (declusterizado) en el test?
cell_had_ms = (y_2d.sum(axis=0) > 0).astype(int)
n_ms_cells  = int(cell_had_ms.sum())

auc_esp = roc_auc_score(cell_had_ms, risk_per_cell)
k20       = max(1, int(0.20 * n_active))
top_cells = np.argsort(-risk_per_cell)[:k20]
recall_top20 = cell_had_ms[top_cells].sum() / max(n_ms_cells, 1)
risk_con = risk_per_cell[cell_had_ms == 1].mean()
risk_sin = risk_per_cell[cell_had_ms == 0].mean()

print("=== Validacion cuantitativa del mapa de riesgo (espacial) ===")
print(f"Celdas con mainshock en test:                 {n_ms_cells} de {n_active}")
print(f"AUC espacial (riesgo vs celda-con-mainshock): {auc_esp:.4f}")
print(f"Recall en top-20% de riesgo:                  {recall_top20:.1%}   (azar: 20%)")
print(f"Riesgo medio  CON mainshock:                  {risk_con:.4f}")
print(f"Riesgo medio  SIN mainshock:                  {risk_sin:.4f}   "
      f"(ratio {risk_con / max(risk_sin, 1e-9):.1f}x)")
print()
print("-> El mapa concentra los mainshocks reales en zonas de alto riesgo: valida")
print("   cuantitativamente la coherencia ESPACIAL (la parte del pipeline que SI funciona).")

## Interpretación

El mapa de riesgo (μ) es una superficie continua suavizada de las probabilidades por celda, y el mapa de incertidumbre (σ) es bajo cerca de las celdas con datos y crece en las zonas sin celdas activas, donde el GP extrapola.

Como el modelo espacial iguala a la climatología de Poisson y la AUC temporal por celda es ~0.50, este mapa refleja la estructura espacial estable de la sismicidad (las zonas crónicamente más activas), no la capacidad de anticipar el momento de los terremotos. Es un mapa de peligro estático, útil para la priorización espacial, con la aportación del GP de cuantificar la incertidumbre.

Los mainshocks reales del test (estrellas) tienden a caer en las zonas de μ alto, lo que valida la coherencia espacial del mapa.